# 05 Legacy Counterfactual Full-Year Generation

This notebook shows the older exploratory combined generation workflow for the official hourly test period.

Current scope:

- use actual hourly prices as anchors for counterfactual realized 15-minute paths;
- use the selected hourly forecast candidates as anchors for counterfactual 15-minute forecast inputs;
- fit the selected quarter-hour shape model on the observed post-implementation library;
- generate flat, low-volatility, empirical/medium, high-volatility, and stress realized paths while preserving the hourly mean exactly.

Methodology note:

- these Phase 5 artifacts are legacy exploratory generation outputs;
- the authoritative downstream thesis input is the separately frozen canonical actual path built by the dedicated frozen-actual pipeline.
- thesis-grade bidding and optimisation runs must not use these realized-path outputs as actual market truth.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path
    for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
    if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)

PACKAGE_ROOT = REPO_ROOT / "scripts" / "Data" / "02_Forecasting" / "01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from quarterhour_da import (
    QuarterHourDAExtensionConfig,
    assert_thesis_grade_actual_source_authorized,
    build_thesis_grade_frozen_actual_metadata,
    find_frozen_actual_version,
    find_latest_canonical_actual_run,
    find_latest_observed_deterministic_run,
    find_latest_phase01_run,
    find_latest_phase02_run,
    find_latest_phase03_run,
    find_latest_phase04_run,
    find_latest_phase07_run,
    find_latest_phase07_upstream_refresh_run,
    load_frozen_actual_diagnostics,
    load_frozen_actual_manifest,
    load_frozen_actual_path,
    resolve_frozen_actual_registry_entry,
    run_observed_market_deterministic_forecast,
)

config = QuarterHourDAExtensionConfig()
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
from quarterhour_da import find_latest_phase05_run

## Optional Phase 5 Runner

In [ ]:
RUN_PHASE05 = False

if RUN_PHASE05:
    command = [
        sys.executable,
        str(REPO_ROOT / "scripts" / "Data" / "02_Forecasting" / "01_DA_prices" / "run_15min_phase05_counterfactual_generation.py"),
    ]
    completed = subprocess.run(command, cwd=REPO_ROOT, capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"Phase 5 counterfactual generation failed with exit code {completed.returncode}.")

In [ ]:
latest_run = find_latest_phase05_run(config)
if latest_run is None:
    raise FileNotFoundError("No saved Phase 5 artifact exists yet. Run the phase 5 script first.")

generation_summary = pd.read_csv(latest_run / "counterfactual_generation_summary.csv")
backoff_summary = pd.read_csv(latest_run / "counterfactual_backoff_summary.csv")
coverage_summary = pd.read_csv(latest_run / "hourly_forecast_coverage_summary.csv")
shape_model_summary = pd.read_csv(latest_run / "shape_model_for_counterfactual_summary.csv")
validation_checks = pd.read_csv(latest_run / "validation_checks.csv")
run_summary = json.loads((latest_run / "run_summary.json").read_text(encoding="utf-8"))

display(pd.DataFrame([{"latest_phase05_run": str(latest_run)}]))

## Shape Model Used For Counterfactual Forecast Inputs

In [ ]:
display(shape_model_summary)

## Counterfactual Generation Summary

In [ ]:
display(generation_summary)

## Hourly Forecast Coverage Summary

In [ ]:
display(coverage_summary)

## Sampler Backoff Summary

In [ ]:
display(backoff_summary)

## Validation Checks

In [ ]:
display(validation_checks)